# Interface PINNs (I-PINNs): A physics-informed neural networks framework for interface problems

**Paper:** Sarma, A.K., Roy, S., Annavarapu, C., Roy, P., Jagannathan, S. (2024). *Interface PINNs (I-PINNs): A physics-informed neural networks framework for interface problems.* Computer Methods in Applied Mechanics and Engineering, 429, 117135.

**Carpeta origen:** `PINNs/3. Arquitecturas, frameworks y variantes/Interface PINNs (I-PINNs) Aphysics-informed neural networks.pdf`

## Como se usan las PINNs en este paper

El paper muestra primero que una PINN convencional (una unica red con activacion suave global) **no puede representar discontinuidades fuertes o debiles** en la solucion o su derivada (Fig. 3), porque el Teorema de Aproximacion Universal solo garantiza aproximar funciones continuas. Su solucion, **I-PINNs**, es arquitectonicamente muy simple pero efectiva: para un dominio dividido en subdominios $\Omega_m$ separados por interfaces, se usa **una unica red compartida** (mismos pesos y sesgos $\mathbf{w},\mathbf{b}$ en todos los subdominios) pero con una **funcion de activacion distinta por subdominio** (Eq. 12):

$$u^\theta(x)=\begin{cases}\sigma_1(\mathbf{w}^T x+\mathbf{b}) & \text{en }\Omega_1\\ \sigma_2(\mathbf{w}^T x+\mathbf{b}) & \text{en }\Omega_2\end{cases}$$

A diferencia de XPINNs o M-PINN (que usan subredes **completamente independientes**, con muchos mas parametros), I-PINNs solo diversifica la no-linealidad, logrando (segun el paper) precision RMSE **al menos dos ordenes de magnitud mejor** que una PINN convencional, a una decima parte del costo computacional. La perdida total (Eq. 15-16) combina el residuo de la EDP, las condiciones de contorno, y las condiciones de salto en la interfaz (continuidad de la variable primaria y del flujo):

$$\zeta(\theta)=MSE_{eq}+MSE_{bc}^d+MSE_{bc}^n+MSE_{ic}^d+MSE_{ic}^n$$

Este cuaderno reproduce fielmente el **primer ejemplo 1D del paper** (Seccion 3.1.1): la ecuacion de Poisson con coeficientes discontinuos y **tres interfaces materiales** en $x=0.25,0.5,0.75$ (Eq. 20), comparando una I-PINN (pesos compartidos, activacion por subdominio) contra una PINN convencional (una unica red/activacion global).

## Repositorio publico de referencia

El PDF no incluye un repositorio de codigo propio, ni se encontro uno especifico al buscar en GitHub. Como referencia general del framework PINN base:

- **maziarraissi/PINNs** &mdash; https://github.com/maziarraissi/PINNs

In [ ]:
# Instalacion de dependencias (ejecutar si no estan ya instaladas en el entorno)
%pip install -q torch numpy matplotlib

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as Fnn
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 1. Ecuacion de Poisson 1D con 3 interfaces materiales (Eq. 20) y solucion exacta (cerrada por tramos)

In [ ]:
kappa = [1.0, 0.1, 0.5, 0.75]   # kappa_1..kappa_4
f_src = 1.0                      # f_m = 1 para todas las regiones
boundaries = [0.0, 0.25, 0.5, 0.75, 1.0]

def subdomain_of(x):
    idx = np.searchsorted(boundaries, x, side='right') - 1
    return np.clip(idx, 0, 3)

# u_m(x) = (f/(2*kappa_m)) x^2 + A_m x + B_m. Se arma el sistema lineal 8x8 para
# (A1,B1,A2,B2,A3,B3,A4,B4) a partir de BCs y condiciones de salto (Eq. 20).
M = np.zeros((8, 8))
rhs = np.zeros(8)

def u_row(m, x):
    row = np.zeros(8)
    row[2*m] = x        # coef. de A_m
    row[2*m+1] = 1.0     # coef. de B_m
    return row, (f_src / (2 * kappa[m])) * x**2

def du_row(m, x):
    row = np.zeros(8)
    row[2*m] = kappa[m]  # kappa_m * u_m'(x) = kappa_m*(x/kappa_m + A_m) = x + kappa_m*A_m
    return row, x

r = 0
row, c = u_row(0, 0.0); M[r] = row; rhs[r] = -c; r += 1                      # u1(0)=0
row, c = u_row(3, 1.0); M[r] = row; rhs[r] = -c; r += 1                      # u4(1)=0
for m, xi in zip([0, 1, 2], [0.25, 0.5, 0.75]):
    row1, c1 = u_row(m, xi); row2, c2 = u_row(m + 1, xi)
    M[r] = row1 - row2; rhs[r] = c2 - c1; r += 1                             # [[u]]=0
    row1, c1 = du_row(m, xi); row2, c2 = du_row(m + 1, xi)
    M[r] = row1 - row2; rhs[r] = c2 - c1; r += 1                             # [[kappa u']]=0

coeffs = np.linalg.solve(M, rhs)
A = coeffs[0::2]
B = coeffs[1::2]

def exact_u(x):
    x = np.atleast_1d(x)
    m = subdomain_of(x)
    return (f_src / (2 * np.array(kappa)[m])) * x**2 + A[m] * x + B[m]

x_plot = np.linspace(0, 1, 400)
plt.figure(figsize=(7, 3))
plt.plot(x_plot, exact_u(x_plot))
for b in boundaries[1:-1]:
    plt.axvline(b, color='gray', linestyle=':')
plt.xlabel('x'); plt.ylabel('u(x)'); plt.title('Solucion exacta (4 subdominios, kappa discontinua)')
plt.show()

## 2. I-PINN: pesos y sesgos COMPARTIDOS, activacion distinta por subdominio (Eq. 12, Fig. 5)

In [ ]:
def swish(x):
    return x * torch.sigmoid(x)

ACTIVATIONS = [torch.tanh, torch.sigmoid, swish, Fnn.elu]  # una por subdominio (Seccion 3)


class IPINN(nn.Module):
    """Mismas capas lineales para todos los subdominios; solo cambia la activacion (Eq. 12)."""
    def __init__(self, n_neurons=30):
        super().__init__()
        self.lin1 = nn.Linear(1, n_neurons)
        self.lin2 = nn.Linear(n_neurons, n_neurons)
        self.lin3 = nn.Linear(n_neurons, 1)

    def forward(self, x, sub_idx):
        h = self.lin1(x)
        h_act = torch.zeros_like(h)
        for m in range(4):
            mask = sub_idx == m
            if mask.any():
                h_act[mask] = ACTIVATIONS[m](h[mask])
        h2 = self.lin2(h_act)
        h2_act = torch.zeros_like(h2)
        for m in range(4):
            mask = sub_idx == m
            if mask.any():
                h2_act[mask] = ACTIVATIONS[m](h2[mask])
        return self.lin3(h2_act)


class ConventionalPINN(nn.Module):
    """Baseline: una unica red con activacion suave global (sin decomposicion)."""
    def __init__(self, n_neurons=30):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(1, n_neurons), nn.Tanh(),
                                  nn.Linear(n_neurons, n_neurons), nn.Tanh(),
                                  nn.Linear(n_neurons, 1))

    def forward(self, x, sub_idx=None):
        return self.net(x)


def d_dx(f, x):
    return torch.autograd.grad(f, x, grad_outputs=torch.ones_like(f),
                                create_graph=True, retain_graph=True)[0]

## 3. Perdida (Eq. 16): residuo PDE por subdominio + contorno + salto de $u$ y flujo en las 3 interfaces

In [ ]:
N_per_sub = 100
x_col_np = np.concatenate([np.random.uniform(boundaries[m] + 1e-3, boundaries[m + 1] - 1e-3, N_per_sub)
                            for m in range(4)])
sub_col_np = subdomain_of(x_col_np)
x_col = torch.tensor(x_col_np, dtype=torch.float32, device=device).view(-1, 1).requires_grad_(True)
sub_col = torch.tensor(sub_col_np, dtype=torch.long, device=device)
kappa_col = torch.tensor(np.array(kappa)[sub_col_np], dtype=torch.float32, device=device).view(-1, 1)

x0 = torch.zeros(1, 1, device=device); sub0 = torch.zeros(1, dtype=torch.long, device=device)
x1 = torch.ones(1, 1, device=device); sub1 = torch.full((1,), 3, dtype=torch.long, device=device)

interface_x = [0.25, 0.5, 0.75]


def compute_loss(model, lam_pde=1.0, lam_bc=100.0, lam_ic=100.0):
    u = model(x_col, sub_col)
    du = d_dx(u, x_col)
    d2u = d_dx(kappa_col * du, x_col)
    loss_pde = torch.mean((d2u - f_src)**2)

    u0 = model(x0, sub0); u1_ = model(x1, sub1)
    loss_bc = u0.pow(2).mean() + u1_.pow(2).mean()

    loss_ic = 0.0
    for m, xi in enumerate(interface_x):
        xl = torch.tensor([[xi]], dtype=torch.float32, device=device, requires_grad=True)
        xr = torch.tensor([[xi]], dtype=torch.float32, device=device, requires_grad=True)
        sl = torch.full((1,), m, dtype=torch.long, device=device)
        sr = torch.full((1,), m + 1, dtype=torch.long, device=device)
        ul = model(xl, sl); ur = model(xr, sr)
        dul = d_dx(ul, xl); dur = d_dx(ur, xr)
        loss_ic = loss_ic + (ul - ur).pow(2).mean() + (kappa[m] * dul - kappa[m + 1] * dur).pow(2).mean()

    return lam_pde * loss_pde + lam_bc * loss_bc + lam_ic * loss_ic

## 4. Entrenamiento: I-PINN vs PINN convencional

In [ ]:
def train(model, epochs=4000, lr=1e-3):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    hist = []
    for epoch in range(epochs):
        opt.zero_grad()
        loss = compute_loss(model)
        loss.backward()
        opt.step()
        hist.append(loss.item())
        if epoch % 1000 == 0:
            print(f'epoch {epoch:5d} | loss={loss.item():.4e}')
    return hist


print('--- Entrenando I-PINN ---')
model_ipinn = IPINN().to(device)
hist_ipinn = train(model_ipinn)

print('--- Entrenando PINN convencional ---')
model_conv = ConventionalPINN().to(device)
hist_conv = train(model_conv)

## 5. Resultados: RMSE (cf. Eq. 17 y Tabla de resultados 1D del paper)

In [ ]:
x_test_np = np.linspace(0, 1, 300)
sub_test_np = subdomain_of(x_test_np)
x_test = torch.tensor(x_test_np, dtype=torch.float32, device=device).view(-1, 1)
sub_test = torch.tensor(sub_test_np, dtype=torch.long, device=device)

with torch.no_grad():
    u_ipinn = model_ipinn(x_test, sub_test).cpu().numpy().flatten()
    u_conv = model_conv(x_test, sub_test).cpu().numpy().flatten()
u_ex = exact_u(x_test_np)

rmse_ipinn = np.sqrt(np.mean((u_ipinn - u_ex)**2))
rmse_conv = np.sqrt(np.mean((u_conv - u_ex)**2))

plt.figure(figsize=(8, 4.5))
plt.plot(x_test_np, u_ex, label='Exacta', linewidth=2)
plt.plot(x_test_np, u_ipinn, '--', label=f'I-PINN (RMSE={rmse_ipinn:.2e})')
plt.plot(x_test_np, u_conv, ':', label=f'PINN convencional (RMSE={rmse_conv:.2e})')
for b in boundaries[1:-1]:
    plt.axvline(b, color='gray', linestyle=':', alpha=0.5)
plt.xlabel('x'); plt.ylabel('u(x)')
plt.title('I-PINN vs PINN convencional: Poisson con 3 interfaces materiales')
plt.legend()
plt.show()

print(f'RMSE I-PINN:            {rmse_ipinn:.4e}')
print(f'RMSE PINN convencional: {rmse_conv:.4e}')